# Holographic Entanglement Entropy in Vaidya-AdS
## Comparing Affine vs. Local Coordinate Parameterizations

This notebook calculates the regularized proper length of spacelike geodesics (which correspond to minimal surfaces computing holographic entanglement entropy) in a 3-dimensional Vaidya-AdS spacetime. 

We will compare two methods of integrating the geodesic equations:
1. **Affine Parameterization ($\lambda$):** Using an external parameter (via `HoloML`).
2. **Local Coordinate Parameterization ($x$):** Using the boundary spatial coordinate $x$ as the parameter along the curve.

### 1. The Vaidya-AdS Background and Mass Profile

We work in Eddington-Finkelstein (EF) coordinates $(v, r, x)$, where the metric for an asymptotically AdS$_3$ spacetime is given by:

$$ds^2 = -f(r,v) dv^2 + 2 dv dr + r^2 dx^2$$

The blackening function $f(r,v)$ determines the location of the horizon and is defined as:

$$f(r,v) = r^2 - m(v)$$

where $m(v)$ is the time-dependent mass profile of the infalling shell. We model the collapse using a smooth hyperbolic tangent function:

$$m(v) = \frac{m_0 + 1}{2} \tanh\left(\frac{v}{v_s}\right) + \frac{m_0 - 1}{2}$$

This describes a transition from a pure AdS state (or a lighter black hole) to a heavier black hole of mass $m_0$ over a timescale $v_s$.

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import jax
import jax.numpy as jnp
import plotly.graph_objects as go
import sys
import os

print(os.getcwd())
os.chdir("")

# Ensure we can import HoloML code
# Update this path to where your HoloML directory is located
sys.path.append(os.path.abspath("HoloML/Vaidya_AdS/src"))
try:
    from Vaidya_AdS import integrate_geodesic as holoml_integrate, ds_dlambda
except ImportError:
    print("Warning: HoloML module not found. The affine comparison won't execute.")

# --- SHARED PHYSICS: MASS PROFILE ---
m0, vs = 1.0, 1.0

def m_vaidya(v): 
    return (m0 + 1) / 2 * np.tanh(v / vs) + (m0 - 1) / 2

def dm_dv_vaidya(v): 
    return (m0 + 1) / (2 * vs) * (1 / np.cosh(v / vs))**2

/home/paulhan65/holography/consistency_checks


### 2. Geodesic Equations in Local Coordinates ($x$)

Instead of an affine parameter $\lambda$, we parameterize the geodesic using the spatial coordinate $x$. Therefore, we are looking for the functions $r(x)$ and $v(x)$. Let primes denote derivatives with respect to $x$ (e.g., $v' = \frac{dv}{dx}$).

By rearranging the standard affine geodesic equations and enforcing that $x$ is our parameter, we obtain the following coupled second-order ordinary differential equations (ODEs):

$$v'' = \frac{r^2 - r^2 (v')^2 + 2 v' r'}{r}$$

$$r'' = \frac{4 \frac{r^3}{r_*^2} r' - 2 r r' + \frac{\partial f}{\partial r} r' (v')^2 + \frac{\partial f}{\partial v} (v')^3 + 2(f v' - r')v''}{2 v'}$$

**Note on the Turning Point:** At the deepest point in the bulk (the turning point $r_*$), the surface is smooth and horizontal, meaning $v' = 0$ and $r' = 0$. This creates a $0/0$ singularity in the $r''$ equation. By applying L'Hôpital's rule or expanding the equations around the turning point, the limit evaluates cleanly to:

$$r''(0) = r_* f(r_*, v_{turn}) = r_* (r_*^2 - m(v_{turn}))$$

In [ ]:
# --- LOCAL COORDINATE X DERIVATIVES ---
def local_derivatives(x, state, r_star):
    r, v, rp, vp = state
    
    # f(r,v) and its derivatives
    f_val = r**2 - m_vaidya(v)
    df_dr = 2 * r
    df_dv = -dm_dv_vaidya(v)
    
    # Geodesic equations parameterized by x
    vpp = (r**2 - r**2 * vp**2 + 2 * vp * rp) / r
    
    num = (4 * (r**3 / r_star**2) * rp - 2 * r * rp 
           + df_dr * rp * vp**2 + df_dv * vp**3 + 2 * (f_val * vp - rp) * vpp)
    
    # Handle the turning point singularity smoothly
    if abs(vp) > 1e-10:
        rpp = num / (2 * vp)
    else:
        rpp = f_val * r
        
    return [rp, vp, rpp, vpp]

### 3. Proper Length Integration

To find the entanglement entropy, we must compute the proper length $L$ of the spacelike geodesic. Using the metric, the line element squared along the curve $r(x), v(x)$ is:

$$ds^2 = \left( -f(r,v) (v')^2 + 2 v' r' + r^2 \right) dx^2$$

The total proper length from the turning point out to the boundary cutoff $r_{cut}$ (for both halves of the symmetric geodesic) is given by integrating $\frac{ds}{dx}$:

$$L = 2 \int_{0}^{x_{bdy}} \sqrt{-f(r,v) (v')^2 + 2 v' r' + r^2} \ dx$$

In [ ]:
def local_proper_length(x_vals, r_vals, v_vals, rp_vals, vp_vals):
    # Calculate f(r,v) along the trajectory
    f_vals = r_vals**2 - np.array([m_vaidya(v) for v in v_vals])
    
    # Calculate ds/dx
    inside = -f_vals * vp_vals**2 + 2.0 * vp_vals * rp_vals + r_vals**2
    ds_dx = np.sqrt(np.maximum(inside, 0.0))
    
    # Proper length of the full geodesic (both sides integrated via trapezoidal rule)
    return 2.0 * np.trapezoid(ds_dx, x_vals)

### 4. Setting Initial Conditions and Executing the Comparison

Because of the coordinate singularity at the exact turning point $x=0$, we start the numerical integration at a very small offset $x_0 = 10^{-4}$. We determine the initial state using a Taylor expansion around $x=0$:

$$r(x) \approx r_* + \frac{1}{2} r''(0) x^2$$
$$v(x) \approx v_{turn} + \frac{1}{2} v''(0) x^2$$

where $v''(0) = r_*$ and $r''(0) = (r_*^2 - m(v_{turn})) r_*$. The initial derivatives are simply $r'(x) \approx r''(0) x$ and $v'(x) \approx v''(0) x$.

In [ ]:
# --- SYSTEM PARAMETERS ---
v_turn = 0.5
r_star = 2.0
r_cut = 50.0

print("--- Running HoloML Affine Integration ---")
# 1. HoloML Affine Integration
try:
    traj_h = holoml_integrate(r_star, v_turn, n_steps=20000, dt=0.002)
    r_h = np.array(traj_h[:, 1])
    x_h = np.array(traj_h[:, 2])
    v_h = np.array(traj_h[:, 0])

    # Cut off at the boundary
    hit_h = np.where(r_h >= r_cut)[0][0]
    traj_h_cut = traj_h[:hit_h+1]

    # Compute HoloML Length
    sdot_h = jax.vmap(ds_dlambda)(traj_h_cut)
    L_holoml = 2.0 * 0.002 * (0.5 * sdot_h[0] + np.sum(sdot_h[1:-1]) + 0.5 * sdot_h[-1])
except NameError:
    print("HoloML functions not available. Skipping affine integration.")
    L_holoml = 0
    x_h = np.zeros(10)
    hit_h = 0

print("\n--- Running Local Coordinate Integration ---")
# 2. Local Coordinate Integration
x0 = 1e-4

# Taylor expansion for initial conditions
r2_init = (r_star**2 - m_vaidya(v_turn)) * r_star
v2_init = r_star

state0 = [
    r_star + 0.5 * r2_init * x0**2,  # r(x0)
    v_turn + 0.5 * v2_init * x0**2,  # v(x0)
    r2_init * x0,                    # r'(x0)
    v2_init * x0                     # v'(x0)
]

# Event function to stop integration when the boundary r_cut is reached
def hit_boundary(x, state, r_star):
    return state[0] - r_cut
hit_boundary.terminal = True

sol_l = solve_ivp(
    local_derivatives, 
    [x0, 2.0], 
    state0, 
    args=(r_star,), 
    events=hit_boundary, 
    rtol=1e-10, 
    atol=1e-10
)

x_l, r_l, v_l = sol_l.t, sol_l.y[0], sol_l.y[1]
rp_l, vp_l = sol_l.y[2], sol_l.y[3]

L_local = local_proper_length(x_l, r_l, v_l, rp_l, vp_l)

# --- OUTPUT STATS ---
print(f"\nComparison: v_turn={v_turn}, r_star={r_star}")
print(f"{'Metric':<15} | {'HoloML (Affine)':<18} | {'Local (Coord x)':<18} | {'Diff'}")
print("-" * 75)
print(f"{'Total Length L':<15} | {L_holoml:<18.8f} | {L_local:<18.8f} | {abs(L_holoml-L_local):.2e}")
print(f"{'Boundary h':<15} | {x_h[hit_h]:<18.8f} | {x_l[-1]:<18.8f} | {abs(x_h[hit_h]-x_l[-1]):.2e}")

### 5. 3D Visualization of the Geodesics

Finally, we visualize the resulting curves. We plot both the Affine integration (HoloML) and the Local Coordinate ($x$) integration in the 3D bulk space $(X, Y, v)$, overlaying them on top of the dynamic apparent horizon. To map coordinates into a visually bounded domain, we apply a compactification transformation $R = \arctan(r)$.

In [2]:
# --- PLOTLY VISUALIZATION ---
def get_plot_coords(x, r, v):
    # Compactify the radial coordinate for visualization
    R = np.arctan(r)
    # Mirror the coordinates to show the full geodesic (both sides of turning point)
    xf = np.concatenate([-x[::-1], x])
    rf = np.concatenate([R[::-1], R])
    vf = np.concatenate([v[::-1], v])
    return rf * np.cos(xf), rf * np.sin(xf), vf

# Get coordinates for both methods
try:
    Xh, Yh, Zh = get_plot_coords(x_h[:hit_h+1], r_h[:hit_h+1], v_h[:hit_h+1])
except NameError:
    Xh, Yh, Zh = [], [], []

Xl, Yl, Zl = get_plot_coords(x_l, r_l, v_l)

fig = go.Figure()

# Plot HoloML trajectory
if len(Xh) > 0:
    fig.add_trace(go.Scatter3d(
        x=Xh, y=Yh, z=Zh, 
        mode='lines', 
        name='HoloML (Affine)', 
        line=dict(color='red', width=8)
    ))

# Plot Local Coordinate trajectory
fig.add_trace(go.Scatter3d(
    x=Xl, y=Yl, z=Zl, 
    mode='lines', 
    name='Local (Coord x)', 
    line=dict(color='cyan', width=4)
))

# Plot the Apparent Horizon
v_range = np.linspace(-1, 2, 50)
theta_range = np.linspace(0, 2*np.pi, 50)
V_mesh, TH_mesh = np.meshgrid(v_range, theta_range)

# Horizon occurs where f(r,v) = 0 -> r = sqrt(m(v))
Rh_mesh = np.arctan(np.sqrt(np.maximum(m_vaidya(V_mesh), 0)))

fig.add_trace(go.Surface(
    x=Rh_mesh*np.cos(TH_mesh), 
    y=Rh_mesh*np.sin(TH_mesh), 
    z=V_mesh, 
    colorscale='Greys', 
    opacity=0.2, 
    showscale=False, 
    name='Horizon'
))

fig.update_layout(
    title="Geodesic Comparison: HoloML vs Local Parameterization", 
    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='v')
)

# Render interactive plot
fig.show()


NameError: name 'x_l' is not defined